# ML-KEM (Kyber-768) PYNQ Driver
## Python driver to control the 3 HLS accelerator IPs on Kria KR260 via PYNQ

Each kernel (**KeyGen**, **Encaps**, **Decaps**) is a separate overlay with its own `.bit` + `.hwh` pair.

The driver handles:
- DMA buffer allocation (`pynq.allocate`)
- AXI-Lite register programming (pointer + control)
- Start / poll-done / read-back

Register maps extracted from the `.hwh` files (Vivado block design).

## 1. Import Libraries & Check PYNQ Availability

In [ ]:
import numpy as np
import time
import os

# Try importing pynq - will only work on the actual FPGA board
try:
    from pynq import Overlay, allocate
    PYNQ_AVAILABLE = True
    print("[INFO] PYNQ library loaded successfully!")
except ImportError:
    PYNQ_AVAILABLE = False
    print("[WARN] pynq not available. Running in simulation/offline mode.")

## 2. Constants — Kyber-768 Parameters

In [ ]:
# ===========================================================
# Constants - Kyber-768
# ===========================================================
PK_SIZE   = 1184   # Public key bytes
SK_SIZE   = 2400   # Secret key bytes (s || pk || H(pk) || z)
CT_SIZE   = 1088   # Ciphertext bytes
SS_SIZE   = 32     # Shared secret bytes
SEED_SIZE = 32     # seed_d / seed_z / randomness_m bytes

# ===========================================================
# AXI-Lite Register Offsets (common to all 3 IPs)
# ===========================================================
REG_CTRL = 0x00    # AP_START(bit0), AP_DONE(bit1), AP_IDLE(bit2), AP_READY(bit3)
REG_GIER = 0x04    # Global Interrupt Enable
REG_IER  = 0x08    # IP Interrupt Enable
REG_ISR  = 0x0C    # IP Interrupt Status

# AP_CTRL bit masks
AP_START = 0x01
AP_DONE  = 0x02
AP_IDLE  = 0x04
AP_READY = 0x08

print("Constants loaded:")
print(f"  PK_SIZE   = {PK_SIZE} bytes")
print(f"  SK_SIZE   = {SK_SIZE} bytes")
print(f"  CT_SIZE   = {CT_SIZE} bytes")
print(f"  SS_SIZE   = {SS_SIZE} bytes")
print(f"  SEED_SIZE = {SEED_SIZE} bytes")

## 3. Helper Functions

In [ ]:
def _write_pointer(ip, offset_lo, addr):
    """Write a 64-bit physical address into two 32-bit AXI-Lite registers."""
    ip.write(offset_lo, addr & 0xFFFFFFFF)
    ip.write(offset_lo + 4, (addr >> 32) & 0xFFFFFFFF)


def _wait_done(ip, timeout_sec=10.0):
    """Poll CTRL register until AP_DONE is asserted."""
    start = time.time()
    while True:
        ctrl = ip.read(REG_CTRL)
        if ctrl & AP_DONE:
            return
        if time.time() - start > timeout_sec:
            raise TimeoutError(
                f"HLS IP did not finish within {timeout_sec}s. "
                f"CTRL=0x{ctrl:08X}"
            )
        time.sleep(0.0001)  # 100us poll interval


def _bytes_to_u64_array(data: bytes) -> np.ndarray:
    """Convert raw bytes (must be multiple of 8) to uint64 array for seed input."""
    assert len(data) % 8 == 0, f"Data length {len(data)} must be multiple of 8"
    return np.frombuffer(data, dtype=np.uint64).copy()


def _check_pynq():
    """Raise error if pynq is not available."""
    if not PYNQ_AVAILABLE:
        raise RuntimeError(
            "pynq library is not installed. "
            "This driver must run on the Kria KR260 board with PYNQ."
        )

print("Helper functions defined: _write_pointer, _wait_done, _bytes_to_u64_array, _check_pynq")

## 4. KeyGen Driver

Register map (from `ml_kem_keygen_0.hwh`):

| Offset | Register |
|--------|----------|
| `0x10-0x14` | `seed_d` pointer (64-bit) |
| `0x1C-0x20` | `seed_z` pointer (64-bit) |
| `0x28-0x2C` | `pk_out` pointer (64-bit) |
| `0x34-0x38` | `sk_out` pointer (64-bit) |

**Inputs:** `seed_d` (32 bytes), `seed_z` (32 bytes)  
**Outputs:** `pk` (1184 bytes), `sk` (2400 bytes)

In [ ]:
# KeyGen Register Offsets
KEYGEN_REG_SEED_D = 0x10
KEYGEN_REG_SEED_Z = 0x1C
KEYGEN_REG_PK_OUT = 0x28
KEYGEN_REG_SK_OUT = 0x34


class MLKEMKeygen:
    """
    ML-KEM KeyGen accelerator driver.

    Inputs:  seed_d (32 bytes), seed_z (32 bytes)
    Outputs: pk (1184 bytes), sk (2400 bytes)
    """

    def __init__(self, bitstream_path: str):
        _check_pynq()
        self.ol = Overlay(bitstream_path)
        self.ip = self.ol.ml_kem_keygen_0

    def run(self, seed_d: bytes, seed_z: bytes, timeout: float = 10.0):
        """
        Run KeyGen on FPGA.

        Args:
            seed_d: 32-byte random seed d
            seed_z: 32-byte random seed z
            timeout: Max seconds to wait for completion

        Returns:
            (pk, sk): tuple of bytes (1184, 2400)
        """
        assert len(seed_d) == SEED_SIZE, f"seed_d must be {SEED_SIZE} bytes"
        assert len(seed_z) == SEED_SIZE, f"seed_z must be {SEED_SIZE} bytes"

        # Allocate contiguous DMA buffers
        seed_d_buf = allocate(shape=(4,), dtype=np.uint64)
        seed_z_buf = allocate(shape=(4,), dtype=np.uint64)
        pk_buf = allocate(shape=(PK_SIZE,), dtype=np.uint8)
        sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)

        try:
            # Fill inputs
            seed_d_buf[:] = _bytes_to_u64_array(seed_d)
            seed_z_buf[:] = _bytes_to_u64_array(seed_z)

            # Flush caches
            seed_d_buf.flush()
            seed_z_buf.flush()

            # Program pointer registers
            _write_pointer(self.ip, KEYGEN_REG_SEED_D, seed_d_buf.physical_address)
            _write_pointer(self.ip, KEYGEN_REG_SEED_Z, seed_z_buf.physical_address)
            _write_pointer(self.ip, KEYGEN_REG_PK_OUT, pk_buf.physical_address)
            _write_pointer(self.ip, KEYGEN_REG_SK_OUT, sk_buf.physical_address)

            # Start
            self.ip.write(REG_CTRL, AP_START)

            # Wait for completion
            _wait_done(self.ip, timeout)

            # Invalidate output caches and read
            pk_buf.invalidate()
            sk_buf.invalidate()

            return bytes(pk_buf), bytes(sk_buf)

        finally:
            seed_d_buf.freebuffer()
            seed_z_buf.freebuffer()
            pk_buf.freebuffer()
            sk_buf.freebuffer()

    def close(self):
        """Free overlay resources."""
        if hasattr(self, 'ol'):
            self.ol.free()

print("MLKEMKeygen class defined.")

## 5. Encaps Driver

Register map (from `ml_kem_encaps_0.hwh`):

| Offset | Register |
|--------|----------|
| `0x10-0x14` | `pk_in` pointer (64-bit) |
| `0x1C-0x20` | `randomness_m` pointer (64-bit) |
| `0x28-0x2C` | `ct_out` pointer (64-bit) |
| `0x34-0x38` | `ss_out` pointer (64-bit) |

**Inputs:** `pk` (1184 bytes), `randomness_m` (32 bytes)  
**Outputs:** `ct` (1088 bytes), `ss` (32 bytes)

In [ ]:
# Encaps Register Offsets
ENCAPS_REG_PK_IN  = 0x10
ENCAPS_REG_RAND_M = 0x1C
ENCAPS_REG_CT_OUT = 0x28
ENCAPS_REG_SS_OUT = 0x34


class MLKEMEncaps:
    """
    ML-KEM Encapsulation accelerator driver.

    Inputs:  pk (1184 bytes), randomness_m (32 bytes)
    Outputs: ct (1088 bytes), ss (32 bytes)
    """

    def __init__(self, bitstream_path: str):
        _check_pynq()
        self.ol = Overlay(bitstream_path)
        self.ip = self.ol.ml_kem_encaps_0

    def run(self, pk: bytes, randomness_m: bytes, timeout: float = 10.0):
        """
        Run Encapsulation on FPGA.

        Args:
            pk: 1184-byte public key (from KeyGen)
            randomness_m: 32-byte random message
            timeout: Max seconds to wait for completion

        Returns:
            (ct, ss): tuple of bytes (1088, 32)
        """
        assert len(pk) == PK_SIZE, f"pk must be {PK_SIZE} bytes"
        assert len(randomness_m) == SEED_SIZE, f"randomness_m must be {SEED_SIZE} bytes"

        pk_buf = allocate(shape=(PK_SIZE,), dtype=np.uint8)
        rand_buf = allocate(shape=(SEED_SIZE,), dtype=np.uint8)
        ct_buf = allocate(shape=(CT_SIZE,), dtype=np.uint8)
        ss_buf = allocate(shape=(SS_SIZE,), dtype=np.uint8)

        try:
            # Fill inputs
            pk_buf[:] = np.frombuffer(pk, dtype=np.uint8)
            rand_buf[:] = np.frombuffer(randomness_m, dtype=np.uint8)

            pk_buf.flush()
            rand_buf.flush()

            # Program pointer registers
            _write_pointer(self.ip, ENCAPS_REG_PK_IN, pk_buf.physical_address)
            _write_pointer(self.ip, ENCAPS_REG_RAND_M, rand_buf.physical_address)
            _write_pointer(self.ip, ENCAPS_REG_CT_OUT, ct_buf.physical_address)
            _write_pointer(self.ip, ENCAPS_REG_SS_OUT, ss_buf.physical_address)

            # Start
            self.ip.write(REG_CTRL, AP_START)

            # Wait
            _wait_done(self.ip, timeout)

            ct_buf.invalidate()
            ss_buf.invalidate()

            return bytes(ct_buf), bytes(ss_buf)

        finally:
            pk_buf.freebuffer()
            rand_buf.freebuffer()
            ct_buf.freebuffer()
            ss_buf.freebuffer()

    def close(self):
        """Free overlay resources."""
        if hasattr(self, 'ol'):
            self.ol.free()

print("MLKEMEncaps class defined.")

## 6. Decaps Driver

Register map (from `ml_kem_decaps_0.hwh`):

| Offset | Register |
|--------|----------|
| `0x10-0x14` | `sk_in` pointer (64-bit) |
| `0x1C-0x20` | `ct_in` pointer (64-bit) |
| `0x28-0x2C` | `ss_out` pointer (64-bit) |

**Inputs:** `sk` (2400 bytes), `ct` (1088 bytes)  
**Outputs:** `ss` (32 bytes)

In [ ]:
# Decaps Register Offsets
DECAPS_REG_SK_IN  = 0x10
DECAPS_REG_CT_IN  = 0x1C
DECAPS_REG_SS_OUT = 0x28


class MLKEMDecaps:
    """
    ML-KEM Decapsulation accelerator driver.

    Inputs:  sk (2400 bytes), ct (1088 bytes)
    Outputs: ss (32 bytes)
    """

    def __init__(self, bitstream_path: str):
        _check_pynq()
        self.ol = Overlay(bitstream_path)
        self.ip = self.ol.ml_kem_decaps_0

    def run(self, sk: bytes, ct: bytes, timeout: float = 10.0):
        """
        Run Decapsulation on FPGA.

        Args:
            sk: 2400-byte secret key (from KeyGen)
            ct: 1088-byte ciphertext (from Encaps)
            timeout: Max seconds to wait for completion

        Returns:
            ss: 32-byte shared secret
        """
        assert len(sk) == SK_SIZE, f"sk must be {SK_SIZE} bytes"
        assert len(ct) == CT_SIZE, f"ct must be {CT_SIZE} bytes"

        sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)
        ct_buf = allocate(shape=(CT_SIZE,), dtype=np.uint8)
        ss_buf = allocate(shape=(SS_SIZE,), dtype=np.uint8)

        try:
            # Fill inputs
            sk_buf[:] = np.frombuffer(sk, dtype=np.uint8)
            ct_buf[:] = np.frombuffer(ct, dtype=np.uint8)

            sk_buf.flush()
            ct_buf.flush()

            # Program pointer registers
            _write_pointer(self.ip, DECAPS_REG_SK_IN, sk_buf.physical_address)
            _write_pointer(self.ip, DECAPS_REG_CT_IN, ct_buf.physical_address)
            _write_pointer(self.ip, DECAPS_REG_SS_OUT, ss_buf.physical_address)

            # Start
            self.ip.write(REG_CTRL, AP_START)

            # Wait
            _wait_done(self.ip, timeout)

            ss_buf.invalidate()

            return bytes(ss_buf)

        finally:
            sk_buf.freebuffer()
            ct_buf.freebuffer()
            ss_buf.freebuffer()

    def close(self):
        """Free overlay resources."""
        if hasattr(self, 'ol'):
            self.ol.free()

print("MLKEMDecaps class defined.")

## 7. MLKEMAccelerator — High-Level Wrapper

Convenience class that manages all 3 overlays for a full KEM flow:
**KeyGen → Encaps → Decaps**

> ⚠️ **Note:** Each call reloads the FPGA since each operation uses a separate bitstream.

In [ ]:
class MLKEMAccelerator:
    """
    High-level wrapper that manages all 3 overlays for a full KEM flow.

    Note: Each call to keygen/encaps/decaps reloads the FPGA,
    since each operation uses a separate bitstream.
    """

    def __init__(self, bitstream_dir: str):
        """
        Args:
            bitstream_dir: Directory containing the 3 subdirectories
                           (Keygen/, Encaps/, Decaps/) with .bit + .hwh files.
        """
        self.keygen_bit = os.path.join(bitstream_dir, "Keygen", "ml_kem_keygen_0.bit")
        self.encaps_bit = os.path.join(bitstream_dir, "Encaps", "ml_kem_encaps_0.bit")
        self.decaps_bit = os.path.join(bitstream_dir, "Decaps", "ml_kem_decaps_0.bit")

    def keygen(self, seed_d: bytes, seed_z: bytes):
        """Generate keypair. Returns (pk, sk)."""
        drv = MLKEMKeygen(self.keygen_bit)
        try:
            return drv.run(seed_d, seed_z)
        finally:
            drv.close()

    def encaps(self, pk: bytes, randomness_m: bytes):
        """Encapsulate. Returns (ct, ss)."""
        drv = MLKEMEncaps(self.encaps_bit)
        try:
            return drv.run(pk, randomness_m)
        finally:
            drv.close()

    def decaps(self, sk: bytes, ct: bytes):
        """Decapsulate. Returns ss."""
        drv = MLKEMDecaps(self.decaps_bit)
        try:
            return drv.run(sk, ct)
        finally:
            drv.close()

    def full_flow(self, seed_d: bytes, seed_z: bytes, randomness_m: bytes):
        """
        Run complete KEM: KeyGen → Encaps → Decaps.
        Verifies that shared secrets match.

        Returns:
            dict with pk, sk, ct, ss_encaps, ss_decaps, match
        """
        pk, sk = self.keygen(seed_d, seed_z)
        ct, ss_encaps = self.encaps(pk, randomness_m)
        ss_decaps = self.decaps(sk, ct)

        return {
            "pk": pk,
            "sk": sk,
            "ct": ct,
            "ss_encaps": ss_encaps,
            "ss_decaps": ss_decaps,
            "match": ss_encaps == ss_decaps,
        }

print("MLKEMAccelerator class defined.")

## 8. Demo — Full ML-KEM 768 Flow on FPGA

Run the complete **KeyGen → Encaps → Decaps** pipeline and verify shared secrets match.

> ⚠️ **Requires:** This cell must run on the **Kria KR260** board with PYNQ installed and bitstreams deployed.

### 8.1 Debug — Kiểm tra Overlay và IP trước khi chạy

Chạy cell này trước demo để xác nhận:
1. Overlay load thành công
2. Tên IP đúng
3. Register map đúng
4. AXI ctrl register đọc được

In [ ]:
_check_pynq()

# --- Set path ---
script_dir = os.path.dirname(os.path.abspath("__file__"))
BIT_DIR = os.path.join(script_dir, "..", "bitstream")
keygen_bit = os.path.join(BIT_DIR, "Keygen", "ml_kem_keygen_0.bit")

print(f"Loading overlay: {keygen_bit}")
print(f"  File exists: {os.path.exists(keygen_bit)}")

# Load overlay
ol = Overlay(keygen_bit)
print(f"\n✅ Overlay loaded successfully!")

# List all IPs in the overlay
print(f"\nIP blocks in overlay:")
for ip_name in ol.ip_dict:
    desc = ol.ip_dict[ip_name]
    has_phys = 'phys_addr' in desc
    print(f"  - {ip_name} (has phys_addr: {has_phys})")
    if has_phys:
        try:
            ip = getattr(ol, ip_name)
            print(f"    Type: {type(ip)}")
        except Exception as e:
            print(f"    ⚠️ Could not instantiate: {e}")

# Access the keygen IP
ip = ol.ml_kem_keygen_0

# Read control register
ctrl = ip.read(REG_CTRL)
print(f"\nCTRL register = 0x{ctrl:08X}")
print(f"  AP_START = {bool(ctrl & AP_START)}")
print(f"  AP_DONE  = {bool(ctrl & AP_DONE)}")
print(f"  AP_IDLE  = {bool(ctrl & AP_IDLE)}")
print(f"  AP_READY = {bool(ctrl & AP_READY)}")

# Check register map
print(f"\nRegister read test:")
for name, offset in [("seed_d_1", 0x10), ("seed_d_2", 0x14), 
                      ("seed_z_1", 0x1C), ("seed_z_2", 0x20),
                      ("pk_out_1", 0x28), ("pk_out_2", 0x2C),
                      ("sk_out_1", 0x34), ("sk_out_2", 0x38)]:
    val = ip.read(offset)
    print(f"  {name} (0x{offset:02X}) = 0x{val:08X}")

print(f"\n✅ All registers accessible!")

# Check if IP starts in IDLE state
if ctrl & AP_IDLE:
    print("✅ IP is in IDLE state — ready to run")
else:
    print("⚠️  IP is NOT idle. May need reset. Try reloading the overlay.")

ol.free()

### 8.2 Debug — Chạy KeyGen step-by-step với chi tiết

Cell này chạy KeyGen từng bước để xác định chính xác lỗi ở đâu.

In [ ]:
_check_pynq()

script_dir = os.path.dirname(os.path.abspath("__file__"))
BIT_DIR = os.path.join(script_dir, "..", "bitstream")
keygen_bit = os.path.join(BIT_DIR, "Keygen", "ml_kem_keygen_0.bit")

# Step 1: Load overlay
print("[Step 1] Loading overlay...")
ol = Overlay(keygen_bit)
ip = ol.ml_kem_keygen_0
print(f"  CTRL before anything = 0x{ip.read(REG_CTRL):08X}")

# Step 2: Generate test seeds
print("\n[Step 2] Generating seeds...")
seed_d = os.urandom(32)
seed_z = os.urandom(32)
print(f"  seed_d ({len(seed_d)} bytes): {seed_d[:8].hex()}...")
print(f"  seed_z ({len(seed_z)} bytes): {seed_z[:8].hex()}...")

# Step 3: Allocate DMA buffers
print("\n[Step 3] Allocating DMA buffers...")
seed_d_buf = allocate(shape=(4,), dtype=np.uint64)
seed_z_buf = allocate(shape=(4,), dtype=np.uint64)
pk_buf = allocate(shape=(PK_SIZE,), dtype=np.uint8)
sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)
print(f"  seed_d_buf: phys=0x{seed_d_buf.physical_address:016X}, shape={seed_d_buf.shape}")
print(f"  seed_z_buf: phys=0x{seed_z_buf.physical_address:016X}, shape={seed_z_buf.shape}")
print(f"  pk_buf:     phys=0x{pk_buf.physical_address:016X}, shape={pk_buf.shape}")
print(f"  sk_buf:     phys=0x{sk_buf.physical_address:016X}, shape={sk_buf.shape}")

# Step 4: Fill seed buffers
print("\n[Step 4] Filling seed buffers...")
seed_d_buf[:] = _bytes_to_u64_array(seed_d)
seed_z_buf[:] = _bytes_to_u64_array(seed_z)
print(f"  seed_d_buf values: {[hex(v) for v in seed_d_buf]}")
print(f"  seed_z_buf values: {[hex(v) for v in seed_z_buf]}")

# Step 5: Flush
print("\n[Step 5] Flushing caches...")
seed_d_buf.flush()
seed_z_buf.flush()
print("  Done.")

# Step 6: Program registers
print("\n[Step 6] Programming pointer registers...")
_write_pointer(ip, KEYGEN_REG_SEED_D, seed_d_buf.physical_address)
_write_pointer(ip, KEYGEN_REG_SEED_Z, seed_z_buf.physical_address)
_write_pointer(ip, KEYGEN_REG_PK_OUT, pk_buf.physical_address)
_write_pointer(ip, KEYGEN_REG_SK_OUT, sk_buf.physical_address)

# Verify registers written correctly
print("  Verifying registers:")
for name, offset, expected in [
    ("seed_d", KEYGEN_REG_SEED_D, seed_d_buf.physical_address),
    ("seed_z", KEYGEN_REG_SEED_Z, seed_z_buf.physical_address),
    ("pk_out", KEYGEN_REG_PK_OUT, pk_buf.physical_address),
    ("sk_out", KEYGEN_REG_SK_OUT, sk_buf.physical_address),
]:
    lo = ip.read(offset)
    hi = ip.read(offset + 4)
    readback = (hi << 32) | lo
    match = "✅" if readback == expected else "❌"
    print(f"  {match} {name}: wrote=0x{expected:016X}, read=0x{readback:016X}")

# Step 7: Check CTRL before start
ctrl = ip.read(REG_CTRL)
print(f"\n[Step 7] CTRL before start = 0x{ctrl:08X}")
print(f"  AP_IDLE = {bool(ctrl & AP_IDLE)}")

# Step 8: Start!
print("\n[Step 8] Writing AP_START...")
ip.write(REG_CTRL, AP_START)

# Step 9: Poll with detailed output
print("\n[Step 9] Polling for completion (timeout=30s)...")
start_time = time.time()
last_print = start_time
done = False
for i in range(300000):  # 30s at 100us intervals
    ctrl = ip.read(REG_CTRL)
    now = time.time()
    
    # Print status every second
    if now - last_print >= 1.0:
        elapsed = now - start_time
        print(f"  [{elapsed:.1f}s] CTRL=0x{ctrl:08X}  START={bool(ctrl&1)} DONE={bool(ctrl&2)} IDLE={bool(ctrl&4)} READY={bool(ctrl&8)}")
        last_print = now
    
    if ctrl & AP_DONE:
        elapsed = now - start_time
        print(f"\n  ✅ AP_DONE asserted after {elapsed*1000:.2f} ms!")
        done = True
        break
    
    if now - start_time > 30.0:
        print(f"\n  ❌ TIMEOUT after 30s. Last CTRL=0x{ctrl:08X}")
        break
    
    time.sleep(0.0001)

if done:
    pk_buf.invalidate()
    sk_buf.invalidate()
    pk = bytes(pk_buf)
    sk = bytes(sk_buf)
    print(f"\n[Result]")
    print(f"  pk ({len(pk)} bytes): {pk[:16].hex()}...")
    print(f"  sk ({len(sk)} bytes): {sk[:16].hex()}...")
    
    # Sanity check: not all zeros
    pk_nonzero = any(b != 0 for b in pk)
    sk_nonzero = any(b != 0 for b in sk)
    print(f"  pk has non-zero data: {'✅' if pk_nonzero else '❌'}")
    print(f"  sk has non-zero data: {'✅' if sk_nonzero else '❌'}")

# Cleanup
seed_d_buf.freebuffer()
seed_z_buf.freebuffer()
pk_buf.freebuffer()
sk_buf.freebuffer()
ol.free()

print("\n[Done] Cleanup complete.")

### 8.3 Debug — Test Encaps riêng (bỏ qua KeyGen)

Dùng **pk** và **randomness_m** từ KAT_768.txt vector #0.
So sánh kết quả **ct** và **ss** với giá trị expected trong KAT.

> Nếu IP vẫn timeout → vấn đề ở bitstream Encaps.
> Nếu chạy xong mà kết quả sai → vấn đề data format.

In [ ]:
# ============================================================
# 8.3 — Test Encaps riêng với KAT vector #0
# ============================================================
_check_pynq()

script_dir = os.path.dirname(os.path.abspath("__file__"))
BIT_DIR = os.path.join(script_dir, "..", "bitstream")
encaps_bit = os.path.join(BIT_DIR, "Encaps", "ml_kem_encaps_0.bit")

# ---- KAT Vector #0: pk, m, ct, ss ----
KAT_PK = bytes.fromhex(
    "a8e651a1e685f22478a8954f007bc7711b930772c78f092e82878e3e937f3679"
    "67532913a8d53dfdf4bfb1f8846746596705cf345142b972a3f16325c40c2952"
    "a37b25897e5ef35fbaeb73a4acbeb6a0b89942ceb195531cfc0a07993954483e"
    "6cbc87c06aa74ff0cac5207e535b260aa98d1198c07da605c4d11020f6c9f7bb"
    "68bb3456c73a01b710bc99d17739a51716aa01660c8b628b2f5602ba65f07ea9"
    "93336e896e83f2c5731bbf03460c5b6c8afecb748ee391e98934a2c57d4d069f"
    "50d88b30d6966f38c37bc649b82634ce7722645ccd625063364646d6d699db57"
    "b45eb67465e16de4d406a818b9eae1ca916a2594489708a43cea88b02a4c03d0"
    "9b44815c97101caf5048bbcb247ae2366cdc254ba22129f45b3b0eb399ca91a3"
    "03402830ec01db7b2ca480cf350409b216094b7b0c3ae33ce10a9124e89651ab"
    "901ea253c8415bd7825f02bb229369af972028f22875ea55af16d3bc69f70c2ee"
    "8b75f28b47dd391f989ade314729c331fa04c1917b278c3eb602868512821adc"
    "825c64577ce1e63b1d9644a612948a3483c7f1b9a258000e30196944a4036276"
    "09c76c7ea6b5de01764d24379117b9ea29848dc555c454bceae1ba5cc72c74ab"
    "96b9c91b910d26b88b25639d4778ae26c7c6151a19c6cd7938454372465e4c5e"
    "c29245acb3db5379de3dabfa629a7c04a8353a8530c95acb732bb4bb81932bb2"
    "ca7a848cd366801444abe23c83b366a87d6a3cf360924c002bae90af65c48060"
    "b3752f2badf1ab2722072554a5059753594e6a702761fc97684c8c4a7540a6b0"
    "7fbc9de87c974aa8809d928c7f4cbbf8045aea5bc667825fd05a521f1a4bf539"
    "210c7113bc37b3e58b0cbfc53c841cbb0371de2e511b989cb7c70c023366d78f"
    "9c37ef047f8720be1c759a8d96b93f65a94114ffaf60d9a81795e995c71152a4"
    "691a5a602a9e1f3599e37c768c7bc108994c0669f3adc957d46b4b6256968e29"
    "0d7892ea85464ee7a750f39c5e3152c2dfc56d8b0c924ba8a959a68096547f66"
    "423c838982a5794b9e1533771331a9a656c28828beb9126a60e95e8c5d906832"
    "c7710705576b1fb9507269ddaf8c95ce9719b2ca8dd112be10bcc9f4a37bd1b1"
    "eeeb33ecda76ae9f69a5d4b2923a86957671d619335be1c4c2c77ce87c41f98a"
    "8cc466460fa300aaf5b301f0a1d09c88e65da4d8ee64f68c02189bbb3584baff"
    "716c85db654048a004333489393a07427cd3e217e6a345f6c2c2b13c27b33727"
    "1c0b27b2dbaa00d237600b5b594e8cf2dd625ea76cf0ed899122c9796b4b0187"
    "004258049a477cd11d68c49b9a0e7b00bce8cac7864cbb375140084744c93062"
    "694ca795c4f40e7acc9c5a1884072d8c38dafb501ee4184dd5a819ec24ec1651"
    "261f962b17a7215aa4a748c15836c389137678204838d7195a85b4f98a1b574c"
    "4cd7909cd1f833effd1485543229d3748d9b5cd6c17b9b3b84aef8bce13e683"
    "733659c79542d615782a71cdeee792bab51bdc4bbfe8308e663144ede8491830"
    "ad98b4634f64aba8b9c042272653920f380c1a17ca87ced7aac41c8288879318"
    "1a6f76e197b7b90ef90943bb3844912911d8551e5466c5767ab0bc61a1a3f736"
    "162ec098a900b12dd8fabbfb3fe8cb1dc4e8315f2af0d32f0017ae136e19f028"
)
KAT_M = bytes.fromhex(
    "eb4a7c66ef4eba2ddb38c88d8bc706b1"
    "d639002198172a7b1942eca8f6c001ba"
)
KAT_CT_EXPECTED = bytes.fromhex(
    "3b835a5fa145387a0819c4daa1e65fbe2ba5400afcd640bbddbbe3585f24bedd"
    "51289694a4fe643cd5af9c8eb277c3f1877a347a97ebea8a037971c6b37993e4"
    "33cfaf580eba4b7fda990d54bf4d60caf9d1cafc477fd956f8e6070b6aeec677"
    "6eb814835407b5f705db9472701d16e00655024a309b14ddbf36d222bb509647"
    "a5a049d5816f49ad9f2975ddb64c2df05ffeb24c6a3f24a786dbf4f6d5666fc5"
    "5fb73539679dc15b72fb4f6ce38feb281d28c908d5195db7008315978ef9d2c6"
    "7dc4dbcc4962467a2d44f7235fa54ebd88bdec32408b1f7aff1b842064075651"
    "f03a3afd2721ed1fe4ff1a8775c6b4d95764555412cff2f8aa4404900f33585f"
    "0bd1b70955cff80130dcc2403920e9744a3d0da914405561ecb2bb32120b7adb"
    "d2f4d8e9a07b4630480b8df8c068934ffd9bc9b855a888eeca090f211905e074"
    "a078ab68917e7445a6c7c7e39403753ce19b6614b9d222ab99f263a681cec6c0"
    "37587ef051f0f7294e376528b31789a530342258241c99ae7d384bcd61012a32"
    "a977c638b09a3bc16a33aa47cf2d7f12d79d8aa50f63c8c53c439800b2ed9bba"
    "9481eb181b4244ed067d62695d6a99dfd7bf8788c159caaf94e9fda92ac5a93f"
    "59a0df7c0f9bbd417cb8cf45d1076006e08a9e585ee4d7394265582a87641f16"
    "53be9edf194401e6e4ee93c4ab054a1b6e81e3bf01fd26f2e9a6db5bf6c0dbd2"
    "1e14c2e1a5a4cff0b267ed95427b0b049eff7fbc093b054510578523ac7a32cc"
    "1f8edfcf078a6c71e6e6788edfda7d7badd375f7d911efafb9cb406e968bc598"
    "9418fb09729ed51c92c4aeae10846384f4a091c405ad85773fe0ade816eddfd6"
    "18ba0ea5deb73cc43592e063015118025542871e7a60f844a6b2c3d630f9c6f8"
    "5791e8d2bdf3578ff92628e8acaf02b88d79797fb1ac30153201fcad2234fbd4"
    "f2fc84fa7d2ab6fb2e4d9b55f11dd91a798726107c6842c3e7a1ca895035a8fe"
    "701058e3426e17bbf04c23e78ffb283e027e1c636b1cf9ded3f5909ebcb0fc63"
    "608e918c9ea9a7f7b6d3ece727dac128d31b7c0ffd9e43046ae6a53c25888d0e"
    "602b2302e255dca8c58c10c010269152582c598fdda0b8f43e311ea15ba96e0d"
    "9ff3936f5f18631fb9d03020e342647be078c12a9475474b3dee55abc0e3dd80"
    "4d73fd929b6af94a67dd27c35b5fc2c9bce500b8103b984423cec746231a5b81"
    "9acdea138816e70a95005ea92f7232b666e772c060f95e20612eb7dad3297a34"
    "2a7817c73e24318a0b761562d1ccb6b5d618cbe06f4b1e7b351b6b831fc83479"
    "eb34bf947b68b3a1b557ad866872656c9f59e7578061e84dbae900af3301bef1"
    "eaa0c6424746302930bb685c8f3d9721521ed61bb648a4d5335c4ebf3061f886"
    "3941955242feeec86462828239f460f55cf9de10bada5627f9d3328362d6ada0"
    "8f70f0c65c5a155b2da66156a6aae555c0371328924928e046135daaf48b86c1"
    "ea78b56f40afb2794fb74b9627e2a43aabf3e17a84ee7ad30cf79eb20a72ac69"
)
KAT_SS_EXPECTED = bytes.fromhex(
    "ac865f839fef1bf3d528dd7504bed2f6"
    "4b5502b0fa81d1c32763658e4aac5037"
)

print(f"KAT pk:  {len(KAT_PK)} bytes (expected {PK_SIZE})")
print(f"KAT m:   {len(KAT_M)} bytes (expected {SEED_SIZE})")
print(f"KAT ct:  {len(KAT_CT_EXPECTED)} bytes (expected {CT_SIZE})")
print(f"KAT ss:  {len(KAT_SS_EXPECTED)} bytes (expected {SS_SIZE})")

# Verify sizes
assert len(KAT_PK) == PK_SIZE, f"pk size mismatch: {len(KAT_PK)} != {PK_SIZE}"
assert len(KAT_M) == SEED_SIZE
assert len(KAT_CT_EXPECTED) == CT_SIZE, f"ct size mismatch: {len(KAT_CT_EXPECTED)} != {CT_SIZE}"
assert len(KAT_SS_EXPECTED) == SS_SIZE

# Step 1: Load Encaps overlay
print("\n[Step 1] Loading Encaps overlay...")
print(f"  Bitstream: {encaps_bit}")
print(f"  File exists: {os.path.exists(encaps_bit)}")
ol = Overlay(encaps_bit)
ip = ol.ml_kem_encaps_0
ctrl = ip.read(REG_CTRL)
print(f"  CTRL = 0x{ctrl:08X} (IDLE={bool(ctrl & AP_IDLE)})")

# Step 2: Allocate DMA buffers
print("\n[Step 2] Allocating DMA buffers...")
pk_buf   = allocate(shape=(PK_SIZE,), dtype=np.uint8)
rand_buf = allocate(shape=(SEED_SIZE,), dtype=np.uint8)
ct_buf   = allocate(shape=(CT_SIZE,), dtype=np.uint8)
ss_buf   = allocate(shape=(SS_SIZE,), dtype=np.uint8)
print(f"  pk_buf:   phys=0x{pk_buf.physical_address:016X}")
print(f"  rand_buf: phys=0x{rand_buf.physical_address:016X}")
print(f"  ct_buf:   phys=0x{ct_buf.physical_address:016X}")
print(f"  ss_buf:   phys=0x{ss_buf.physical_address:016X}")

# Step 3: Fill input buffers with KAT data
print("\n[Step 3] Filling input buffers with KAT data...")
pk_buf[:] = np.frombuffer(KAT_PK, dtype=np.uint8)
rand_buf[:] = np.frombuffer(KAT_M, dtype=np.uint8)
pk_buf.flush()
rand_buf.flush()
print("  Done.")

# Step 4: Program registers
print("\n[Step 4] Programming pointer registers...")
_write_pointer(ip, ENCAPS_REG_PK_IN,  pk_buf.physical_address)
_write_pointer(ip, ENCAPS_REG_RAND_M, rand_buf.physical_address)
_write_pointer(ip, ENCAPS_REG_CT_OUT, ct_buf.physical_address)
_write_pointer(ip, ENCAPS_REG_SS_OUT, ss_buf.physical_address)

# Verify
for name, offset, expected in [
    ("pk_in",  ENCAPS_REG_PK_IN,  pk_buf.physical_address),
    ("rand_m", ENCAPS_REG_RAND_M, rand_buf.physical_address),
    ("ct_out", ENCAPS_REG_CT_OUT, ct_buf.physical_address),
    ("ss_out", ENCAPS_REG_SS_OUT, ss_buf.physical_address),
]:
    lo = ip.read(offset)
    hi = ip.read(offset + 4)
    readback = (hi << 32) | lo
    match = "✅" if readback == expected else "❌"
    print(f"  {match} {name}: wrote=0x{expected:016X}, read=0x{readback:016X}")

# Step 5: Start IP
ctrl = ip.read(REG_CTRL)
print(f"\n[Step 5] CTRL before start = 0x{ctrl:08X}")
print("  Starting IP (AP_START)...")
ip.write(REG_CTRL, AP_START)

# Step 6: Poll (120s timeout)
TIMEOUT_SEC = 120
print(f"\n[Step 6] Polling for completion (timeout={TIMEOUT_SEC}s)...")
start_time = time.time()
last_print = start_time
done = False

while True:
    ctrl = ip.read(REG_CTRL)
    now = time.time()
    elapsed = now - start_time
    
    if now - last_print >= 2.0:
        print(f"  [{elapsed:6.1f}s] CTRL=0x{ctrl:08X}  "
              f"START={bool(ctrl&1)} DONE={bool(ctrl&2)} "
              f"IDLE={bool(ctrl&4)} READY={bool(ctrl&8)}")
        last_print = now
    
    if ctrl & AP_DONE:
        print(f"\n  ✅ AP_DONE after {elapsed:.2f}s ({elapsed*1000:.0f} ms)")
        done = True
        break
    
    if elapsed > TIMEOUT_SEC:
        print(f"\n  ❌ TIMEOUT after {TIMEOUT_SEC}s! CTRL=0x{ctrl:08X}")
        break
    
    time.sleep(0.001)

# Step 7: Check results
if done:
    ct_buf.invalidate()
    ss_buf.invalidate()
    ct_result = bytes(ct_buf)
    ss_result = bytes(ss_buf)
    
    print(f"\n[Step 7] Checking Encaps results against KAT...")
    print(f"  ct size: {len(ct_result)} bytes")
    print(f"  ss size: {len(ss_result)} bytes")
    print(f"  ct first 32B: {ct_result[:32].hex()}")
    print(f"  ss:            {ss_result.hex()}")
    
    ct_match = ct_result == KAT_CT_EXPECTED
    ss_match = ss_result == KAT_SS_EXPECTED
    print(f"\n  ct match KAT: {'✅ PASS' if ct_match else '❌ FAIL'}")
    print(f"  ss match KAT: {'✅ PASS' if ss_match else '❌ FAIL'}")
    
    if not ct_match:
        # Show first difference
        for i in range(len(ct_result)):
            if ct_result[i] != KAT_CT_EXPECTED[i]:
                print(f"  ct first diff at byte {i}: got 0x{ct_result[i]:02X}, expected 0x{KAT_CT_EXPECTED[i]:02X}")
                break
    
    ct_nonzero = any(b != 0 for b in ct_result)
    ss_nonzero = any(b != 0 for b in ss_result)
    print(f"  ct has data: {'✅' if ct_nonzero else '❌ ALL ZEROS'}")
    print(f"  ss has data: {'✅' if ss_nonzero else '❌ ALL ZEROS'}")
else:
    print("\n[Step 7] Skipped — IP did not finish.")
    print("  Encaps IP bị timeout. Vấn đề tương tự KeyGen.")

# Cleanup
pk_buf.freebuffer()
rand_buf.freebuffer()
ct_buf.freebuffer()
ss_buf.freebuffer()
ol.free()
print("\n[Done] Encaps test cleanup complete.")

### 8.4 Debug — Test Decaps riêng (bỏ qua KeyGen)

Dùng **sk** và **ct** từ KAT_768.txt vector #0.
So sánh kết quả **ss** với giá trị expected trong KAT.

> Decaps dùng 3 gmem bundles: `gmem0` (sk_in), `gmem1` (ct_in), `gmem2` (ss_out)

In [ ]:
# ============================================================
# 8.4 — Test Decaps riêng với KAT vector #0
# ============================================================
_check_pynq()

script_dir = os.path.dirname(os.path.abspath("__file__"))
BIT_DIR = os.path.join(script_dir, "..", "bitstream")
decaps_bit = os.path.join(BIT_DIR, "Decaps", "ml_kem_decaps_0.bit")

# ---- KAT Vector #0: sk, ct, ss ----
KAT_SK = bytes.fromhex(
    "da0ac7b660404e613aa1f980380cb36dba18d23256c7267a00a67ba6c2a2b14c"
    "414239662f68bd446c8efdf36656a0891a3cc623fc68b6572f7b29a6de128014"
    "411ee41906d08071f94856e36a832b40338d743516659bd25879c007a52bc958"
    "6f79876afac6c9a30d8fac243bd22425d6adce42ab7ed39014757a958bc8a745"
    "65f019234ff04b34893ed6d05501c37255239aae2ac19f8c75ac5900dae8300d"
    "bba710dc2caae1bca3a38c58342b286b8518f136ad15b9f7bcbb06a5607db375"
    "dbe976457c26c6598257531b2cfb6ee7f51591840804c38388376c27148413da"
    "9e92920bfd9a069e018bd272053da8775c0b739f761db2107cf35a434d69b07e"
    "5bcdb87434138b0cb556761ba522a5747b28747d80eb9d6cc673bee5769377b9"
    "96d36ceb0c0c7ed9a658533324869c18a1a36f31470f14c5ae49ab070507f824"
    "9ce404b49c0a8c3ee42fea9631fa1a0d10d86b93f986e0e3a82e703b74e5ae61"
    "01242421a89aa07fe68588460baa368786486a72e4f24d2dd76cfc03b694a5ba"
    "91a755a0b98f3bf93307c0ab64639aea7a6498a3c3ddc571141abca4678cd2e2"
    "b857fb88f600caa596b44bc422250b2819e0515f0472391853700b01eff9453f"
    "d11876b7c759a07dd845caba4555264a82765193fdf81b620a1e1f923fb24442"
    "cd1cbe94175003ec06ce77a3c64493c199987a300c95c53c0089b5d65c92ea97"
    "1b2ffa93b52a461ea2ac8c199c2f4c2b704297ce3c3949e0735ea8a14aa59e8d"
    "ec0c878399ff70747ab244ce46b5f2230473323d25c66fe6b419b1f4a112e521"
    "4035256bc43ffd2b6b7b378769a6b47000bfb6357d45814baef3857d379e2fb8"
    "b5e5201ab26274bb1b70ad322cd0439b2db109cff0a2f8e600995571ffc38c59"
    "0bc4c7615c69d0c98ef430f30861a77238ffc07061e475d6a30ad1b47fd039c3"
    "a447762db2211dc31d0acacfd55890a5824798f9aead7413dfe028b1012be8b6"
    "ca1026666ac6bc9440a449b51ad8bba7b0921dd4d8b4a578136d1a05db38cc85"
    "8437b25161d1c3c28ee07bbcf2b249110d22781dc3050d8cc0090096b38a8506"
    "96f86e9e6bab325271b2248675011968502881090497fac0af843c1aea76dd81"
    "cf29c012c66227b7f06d9961309b0262f732c9a4d0bbd06727abb8371ff2c118"
    "99a098375c460516b2cc88bcf628ede37d8f3b3342e4490a85606ec03da29b02"
    "56275382a3313dc041114801032c519f350c3e6abac3e33b93b4a19f7c5466e5"
    "8cb1dc14b4a96c475729f971bdf173cdf354824d019427f95b3b4a4a4a958e47"
    "6a6e6991ce6f06cb5dfca7d4380c3d920b5711ac1fcbaf4b9ac800b976d1ec76"
    "6a626cc1900b66b3a9dc62c5c144527a296baf70433bf657c0437f87597bd7c8"
    "bbbe9abc37050931a4a86982a2028a74454c9b810c88d1701c8cc98a1d4ca107"
    "a6b25e962fe4b6b03c95453260b807228637cc9eb12acc0954959a52ae54d197"
    "7300aba0ba2c14609bb28c11d5fac5cac88297603283e867a3648366c724d935"
    "4cd7a196dbd9802f7b88d3fa001f9c9773225462235e91352a20791fd8b87fe3"
    "377ec6a3940b1130a0bb04e7410a34e2580d071d6c56202086787a6590f84393"
    "a8e651a1e685f22478a8954f007bc7711b930772c78f092e82878e3e937f3679"
    "67532913a8d53dfdf4bfb1f8846746596705cf345142b972a3f16325c40c2952"
    "a37b25897e5ef35fbaeb73a4acbeb6a0b89942ceb195531cfc0a07993954483e"
    "6cbc87c06aa74ff0cac5207e535b260aa98d1198c07da605c4d11020f6c9f7bb"
    "68bb3456c73a01b710bc99d17739a51716aa01660c8b628b2f5602ba65f07ea9"
    "93336e896e83f2c5731bbf03460c5b6c8afecb748ee391e98934a2c57d4d069f"
    "50d88b30d6966f38c37bc649b82634ce7722645ccd625063364646d6d699db57"
    "b45eb67465e16de4d406a818b9eae1ca916a2594489708a43cea88b02a4c03d0"
    "9b44815c97101caf5048bbcb247ae2366cdc254ba22129f45b3b0eb399ca91a3"
    "03402830ec01db7b2ca480cf350409b216094b7b0c3ae33ce10a9124e89651ab"
    "901ea253c8415bd7825f02bb229369af972028f22875ea55af16d3bc69f70c2ee"
    "8b75f28b47dd391f989ade314729c331fa04c1917b278c3eb602868512821adc"
    "825c64577ce1e63b1d9644a612948a3483c7f1b9a258000e30196944a4036276"
    "09c76c7ea6b5de01764d24379117b9ea29848dc555c454bceae1ba5cc72c74ab"
    "96b9c91b910d26b88b25639d4778ae26c7c6151a19c6cd7938454372465e4c5e"
    "c29245acb3db5379de3dabfa629a7c04a8353a8530c95acb732bb4bb81932bb2"
    "ca7a848cd366801444abe23c83b366a87d6a3cf360924c002bae90af65c48060"
    "b3752f2badf1ab2722072554a5059753594e6a702761fc97684c8c4a7540a6b0"
    "7fbc9de87c974aa8809d928c7f4cbbf8045aea5bc667825fd05a521f1a4bf539"
    "210c7113bc37b3e58b0cbfc53c841cbb0371de2e511b989cb7c70c023366d78f"
    "9c37ef047f8720be1c759a8d96b93f65a94114ffaf60d9a81795e995c71152a4"
    "691a5a602a9e1f3599e37c768c7bc108994c0669f3adc957d46b4b6256968e29"
    "0d7892ea85464ee7a750f39c5e3152c2dfc56d8b0c924ba8a959a68096547f66"
    "423c838982a5794b9e1533771331a9a656c28828beb9126a60e95e8c5d906832"
    "c7710705576b1fb9507269ddaf8c95ce9719b2ca8dd112be10bcc9f4a37bd1b1"
    "eeeb33ecda76ae9f69a5d4b2923a86957671d619335be1c4c2c77ce87c41f98a"
    "8cc466460fa300aaf5b301f0a1d09c88e65da4d8ee64f68c02189bbb3584baff"
    "716c85db654048a004333489393a07427cd3e217e6a345f6c2c2b13c27b33727"
    "1c0b27b2dbaa00d237600b5b594e8cf2dd625ea76cf0ed899122c9796b4b0187"
    "004258049a477cd11d68c49b9a0e7b00bce8cac7864cbb375140084744c93062"
    "694ca795c4f40e7acc9c5a1884072d8c38dafb501ee4184dd5a819ec24ec1651"
    "261f962b17a7215aa4a748c15836c389137678204838d7195a85b4f98a1b574c"
    "4cd7909cd1f833effd1485543229d3748d9b5cd6c17b9b3b84aef8bce13e683"
    "733659c79542d615782a71cdeee792bab51bdc4bbfe8308e663144ede8491830"
    "ad98b4634f64aba8b9c042272653920f380c1a17ca87ced7aac41c8288879318"
    "1a6f76e197b7b90ef90943bb3844912911d8551e5466c5767ab0bc61a1a3f736"
    "162ec098a900b12dd8fabbfb3fe8cb1dc4e8315f2af0d32f0017ae136e19f028"
    "f57262661358cde8d3ebf990e5fd1d5b896c992ccfaadb5256b68bbf5943b132"
    "b505d7cfad1b497499323c8686325e4792f267aafa3f87ca60d01cb54f29202a"
)
KAT_CT = bytes.fromhex(
    "3b835a5fa145387a0819c4daa1e65fbe2ba5400afcd640bbddbbe3585f24bedd"
    "51289694a4fe643cd5af9c8eb277c3f1877a347a97ebea8a037971c6b37993e4"
    "33cfaf580eba4b7fda990d54bf4d60caf9d1cafc477fd956f8e6070b6aeec677"
    "6eb814835407b5f705db9472701d16e00655024a309b14ddbf36d222bb509647"
    "a5a049d5816f49ad9f2975ddb64c2df05ffeb24c6a3f24a786dbf4f6d5666fc5"
    "5fb73539679dc15b72fb4f6ce38feb281d28c908d5195db7008315978ef9d2c6"
    "7dc4dbcc4962467a2d44f7235fa54ebd88bdec32408b1f7aff1b842064075651"
    "f03a3afd2721ed1fe4ff1a8775c6b4d95764555412cff2f8aa4404900f33585f"
    "0bd1b70955cff80130dcc2403920e9744a3d0da914405561ecb2bb32120b7adb"
    "d2f4d8e9a07b4630480b8df8c068934ffd9bc9b855a888eeca090f211905e074"
    "a078ab68917e7445a6c7c7e39403753ce19b6614b9d222ab99f263a681cec6c0"
    "37587ef051f0f7294e376528b31789a530342258241c99ae7d384bcd61012a32"
    "a977c638b09a3bc16a33aa47cf2d7f12d79d8aa50f63c8c53c439800b2ed9bba"
    "9481eb181b4244ed067d62695d6a99dfd7bf8788c159caaf94e9fda92ac5a93f"
    "59a0df7c0f9bbd417cb8cf45d1076006e08a9e585ee4d7394265582a87641f16"
    "53be9edf194401e6e4ee93c4ab054a1b6e81e3bf01fd26f2e9a6db5bf6c0dbd2"
    "1e14c2e1a5a4cff0b267ed95427b0b049eff7fbc093b054510578523ac7a32cc"
    "1f8edfcf078a6c71e6e6788edfda7d7badd375f7d911efafb9cb406e968bc598"
    "9418fb09729ed51c92c4aeae10846384f4a091c405ad85773fe0ade816eddfd6"
    "18ba0ea5deb73cc43592e063015118025542871e7a60f844a6b2c3d630f9c6f8"
    "5791e8d2bdf3578ff92628e8acaf02b88d79797fb1ac30153201fcad2234fbd4"
    "f2fc84fa7d2ab6fb2e4d9b55f11dd91a798726107c6842c3e7a1ca895035a8fe"
    "701058e3426e17bbf04c23e78ffb283e027e1c636b1cf9ded3f5909ebcb0fc63"
    "608e918c9ea9a7f7b6d3ece727dac128d31b7c0ffd9e43046ae6a53c25888d0e"
    "602b2302e255dca8c58c10c010269152582c598fdda0b8f43e311ea15ba96e0d"
    "9ff3936f5f18631fb9d03020e342647be078c12a9475474b3dee55abc0e3dd80"
    "4d73fd929b6af94a67dd27c35b5fc2c9bce500b8103b984423cec746231a5b81"
    "9acdea138816e70a95005ea92f7232b666e772c060f95e20612eb7dad3297a34"
    "2a7817c73e24318a0b761562d1ccb6b5d618cbe06f4b1e7b351b6b831fc83479"
    "eb34bf947b68b3a1b557ad866872656c9f59e7578061e84dbae900af3301bef1"
    "eaa0c6424746302930bb685c8f3d9721521ed61bb648a4d5335c4ebf3061f886"
    "3941955242feeec86462828239f460f55cf9de10bada5627f9d3328362d6ada0"
    "8f70f0c65c5a155b2da66156a6aae555c0371328924928e046135daaf48b86c1"
    "ea78b56f40afb2794fb74b9627e2a43aabf3e17a84ee7ad30cf79eb20a72ac69"
)
KAT_SS_DECAPS = bytes.fromhex(
    "ac865f839fef1bf3d528dd7504bed2f6"
    "4b5502b0fa81d1c32763658e4aac5037"
)

print(f"KAT sk:  {len(KAT_SK)} bytes (expected {SK_SIZE})")
print(f"KAT ct:  {len(KAT_CT)} bytes (expected {CT_SIZE})")
print(f"KAT ss:  {len(KAT_SS_DECAPS)} bytes (expected {SS_SIZE})")

assert len(KAT_SK) == SK_SIZE, f"sk size mismatch: {len(KAT_SK)} != {SK_SIZE}"
assert len(KAT_CT) == CT_SIZE, f"ct size mismatch: {len(KAT_CT)} != {CT_SIZE}"
assert len(KAT_SS_DECAPS) == SS_SIZE

# Step 1: Load Decaps overlay
print("\n[Step 1] Loading Decaps overlay...")
print(f"  Bitstream: {decaps_bit}")
print(f"  File exists: {os.path.exists(decaps_bit)}")
ol = Overlay(decaps_bit)
ip = ol.ml_kem_decaps_0
ctrl = ip.read(REG_CTRL)
print(f"  CTRL = 0x{ctrl:08X} (IDLE={bool(ctrl & AP_IDLE)})")

# Step 2: Allocate DMA buffers
print("\n[Step 2] Allocating DMA buffers...")
sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)
ct_buf = allocate(shape=(CT_SIZE,), dtype=np.uint8)
ss_buf = allocate(shape=(SS_SIZE,), dtype=np.uint8)
print(f"  sk_buf: phys=0x{sk_buf.physical_address:016X}")
print(f"  ct_buf: phys=0x{ct_buf.physical_address:016X}")
print(f"  ss_buf: phys=0x{ss_buf.physical_address:016X}")

# Step 3: Fill input buffers with KAT data
print("\n[Step 3] Filling input buffers with KAT data...")
sk_buf[:] = np.frombuffer(KAT_SK, dtype=np.uint8)
ct_buf[:] = np.frombuffer(KAT_CT, dtype=np.uint8)
sk_buf.flush()
ct_buf.flush()
print("  Done.")

# Step 4: Program registers
# Decaps register map: sk_in=0x10, ct_in=0x1C, ss_out=0x28
print("\n[Step 4] Programming pointer registers...")
_write_pointer(ip, DECAPS_REG_SK_IN,  sk_buf.physical_address)
_write_pointer(ip, DECAPS_REG_CT_IN,  ct_buf.physical_address)
_write_pointer(ip, DECAPS_REG_SS_OUT, ss_buf.physical_address)

# Verify
for name, offset, expected in [
    ("sk_in",  DECAPS_REG_SK_IN,  sk_buf.physical_address),
    ("ct_in",  DECAPS_REG_CT_IN,  ct_buf.physical_address),
    ("ss_out", DECAPS_REG_SS_OUT, ss_buf.physical_address),
]:
    lo = ip.read(offset)
    hi = ip.read(offset + 4)
    readback = (hi << 32) | lo
    match = "✅" if readback == expected else "❌"
    print(f"  {match} {name}: wrote=0x{expected:016X}, read=0x{readback:016X}")

# Step 5: Start IP
ctrl = ip.read(REG_CTRL)
print(f"\n[Step 5] CTRL before start = 0x{ctrl:08X}")
print("  Starting IP (AP_START)...")
ip.write(REG_CTRL, AP_START)

# Step 6: Poll (120s timeout)
TIMEOUT_SEC = 120
print(f"\n[Step 6] Polling for completion (timeout={TIMEOUT_SEC}s)...")
start_time = time.time()
last_print = start_time
done = False

while True:
    ctrl = ip.read(REG_CTRL)
    now = time.time()
    elapsed = now - start_time
    
    if now - last_print >= 2.0:
        print(f"  [{elapsed:6.1f}s] CTRL=0x{ctrl:08X}  "
              f"START={bool(ctrl&1)} DONE={bool(ctrl&2)} "
              f"IDLE={bool(ctrl&4)} READY={bool(ctrl&8)}")
        last_print = now
    
    if ctrl & AP_DONE:
        print(f"\n  ✅ AP_DONE after {elapsed:.2f}s ({elapsed*1000:.0f} ms)")
        done = True
        break
    
    if elapsed > TIMEOUT_SEC:
        print(f"\n  ❌ TIMEOUT after {TIMEOUT_SEC}s! CTRL=0x{ctrl:08X}")
        break
    
    time.sleep(0.001)

# Step 7: Check results
if done:
    ss_buf.invalidate()
    ss_result = bytes(ss_buf)
    
    print(f"\n[Step 7] Checking Decaps results against KAT...")
    print(f"  ss size: {len(ss_result)} bytes")
    print(f"  ss got:      {ss_result.hex()}")
    print(f"  ss expected: {KAT_SS_DECAPS.hex()}")
    
    ss_match = ss_result == KAT_SS_DECAPS
    print(f"\n  ss match KAT: {'✅ PASS' if ss_match else '❌ FAIL'}")
    
    ss_nonzero = any(b != 0 for b in ss_result)
    print(f"  ss has data: {'✅' if ss_nonzero else '❌ ALL ZEROS'}")
else:
    print("\n[Step 7] Skipped — IP did not finish.")
    print("  Decaps IP bị timeout. Vấn đề tương tự KeyGen.")

# Cleanup
sk_buf.freebuffer()
ct_buf.freebuffer()
ss_buf.freebuffer()
ol.free()
print("\n[Done] Decaps test cleanup complete.")

In [ ]:
# ============================================================
# Configuration — Set the path to your bitstream directory
# ============================================================
# Default: ../bitstream/ relative to this notebook
script_dir = os.path.dirname(os.path.abspath("__file__"))
BIT_DIR = os.path.join(script_dir, "..", "bitstream")

# Or set an absolute path:
# BIT_DIR = "/home/xilinx/bitstreams"

print(f"Bitstream directory: {os.path.abspath(BIT_DIR)}")
print(f"  Keygen: {os.path.exists(os.path.join(BIT_DIR, 'Keygen'))}")
print(f"  Encaps: {os.path.exists(os.path.join(BIT_DIR, 'Encaps'))}")
print(f"  Decaps: {os.path.exists(os.path.join(BIT_DIR, 'Decaps'))}")

In [ ]:
# ============================================================
# Run Full ML-KEM Flow
# ============================================================
_check_pynq()

# Generate random seeds
seed_d = os.urandom(32)
seed_z = os.urandom(32)
rand_m = os.urandom(32)

accel = MLKEMAccelerator(BIT_DIR)

print("ML-KEM 768 FPGA Accelerator Demo")
print("=" * 50)

# ---- KeyGen ----
print("\n[1/3] KeyGen...")
t0 = time.time()
pk, sk = accel.keygen(seed_d, seed_z)
t_keygen = time.time() - t0
print(f"  pk: {len(pk)} bytes")
print(f"  sk: {len(sk)} bytes")
print(f"  Time: {t_keygen*1000:.2f} ms")

# ---- Encaps ----
print("\n[2/3] Encaps...")
t0 = time.time()
ct, ss_enc = accel.encaps(pk, rand_m)
t_encaps = time.time() - t0
print(f"  ct: {len(ct)} bytes")
print(f"  ss: {ss_enc.hex()}")
print(f"  Time: {t_encaps*1000:.2f} ms")

# ---- Decaps ----
print("\n[3/3] Decaps...")
t0 = time.time()
ss_dec = accel.decaps(sk, ct)
t_decaps = time.time() - t0
print(f"  ss: {ss_dec.hex()}")
print(f"  Time: {t_decaps*1000:.2f} ms")

# ---- Verify ----
print("\n" + "=" * 50)
match = ss_enc == ss_dec
print(f"Shared secrets match: {'✅ YES' if match else '❌ NO'}")
print(f"Total time: {(t_keygen + t_encaps + t_decaps)*1000:.2f} ms")